In [ ]:
import random

import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
def set_seed(seed=None, seed_torch=True):
  """
  Function that controls randomness. NumPy and random modules must be imported.

  Args:
    seed : Integer
      A non-negative integer that defines the random state. Default is `None`.
    seed_torch : Boolean
      If `True` sets the random seed for pytorch tensors, so pytorch module
      must be imported. Default is `True`.

  Returns:
    Nothing.
  """
  if seed is None:
    seed = np.random.choice(2 ** 32)
  random.seed(seed)
  np.random.seed(seed)
  if seed_torch:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

  print(f'Random seed {seed} has been set.')


# In case that `DataLoader` is used
def seed_worker(worker_id):
  """
  DataLoader will reseed workers following randomness in
  multi-process data loading algorithm.

  Args:
    worker_id: integer
      ID of subprocess to seed. 0 means that
      the data will be loaded in the main process
      Refer: https://pytorch.org/docs/stable/data.html#data-loading-randomness for more details

  Returns:
    Nothing
  """
  worker_seed = torch.initial_seed() % 2**32
  np.random.seed(worker_seed)
  random.seed(worker_seed)

In [ ]:
def set_device():
  """
  Set the device. CUDA if available, CPU otherwise

  Args:
    None

  Returns:
    Nothing
  """
  device = "cuda" if torch.cuda.is_available() else "cpu"
  if device != "cuda":
    print("GPU is not enabled in this notebook. \n"
          "If you want to enable it, in the menu under `Runtime` -> \n"
          "`Hardware accelerator.` and select `GPU` from the dropdown menu")
  else:
    print("GPU is enabled in this notebook. \n"
          "If you want to disable it, in the menu under `Runtime` -> \n"
          "`Hardware accelerator.` and select `None` from the dropdown menu")

  return device


In [ ]:
SEED = 2021
set_seed(seed=SEED)
DEVICE = set_device()

In [ ]:
#check for gpu
if torch.backends.mps.is_available():
   mps_device = torch.device("mps")
   x = torch.ones(1, device=mps_device)
   print (x)
else:
   print ("MPS device not found.")

In [ ]:
class Net(nn.Module):
    """
    Initialize MLP Network
    """

    def __init__(self, actv, input_feature_num, hidden_unit_nums, output_feature_num, bias=True):
        """
        Initialize MLP Network parameters

        Args:
            actv: string
                Activation function
            input_feature_num: int
                Number of input features
            hidden_unit_nums: list
                Number of units per hidden layer, list of integers
            output_feature_num: int
                Number of output features

        Returns:
            Nothing
        """

        super(Net, self).__init__()
        self.input_feature_num = input_feature_num
        self.mlp = nn.Sequential()

        in_num = input_feature_num
        for i in np.arange(len(hidden_unit_nums)):
            out_num = hidden_unit_nums[i]
            layer = nn.Linear(in_features=in_num, out_features=out_num, bias=bias)
            in_num = out_num
            self.mlp.add_module(f'Linear_{i}', layer)

            actv_layer = eval(f'nn.{actv}')
            self.mlp.add_module(f'Activation_{i}', actv_layer)

        out_layer = nn.Linear(in_features=in_num, out_features=output_feature_num, bias=bias)
        self.mlp.add_module('Output_Linear', out_layer)


    def forward(self, x):
        """
        Simulate forward pass of MLP Network

        Args:
            x: torch.tensor
                Input data
        
        Returns:
            logits: Instance of MLP
                Forward pass of MLP
        """

        x = x.view(-1, self.input_feature_num)
        logits = self.mlp(x)
        
        return logits

In [ ]:
input = torch.zeros((50, 2))
net = Net(actv='LeakyReLU(0.1)', input_feature_num=2, hidden_unit_nums=[100, 10, 5], output_feature_num=3).to(DEVICE)
y = net(input.to(DEVICE))
print(f'The output shape is {y.shape} for an input of shape {input.shape}')

In [ ]:
DEVICE

## dealing with MNIST dataset

In [ ]:
import torchvision
import torchvision.transforms as transforms

batch_size = 4

trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)

testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
len(testset)

In [ ]:
mean = trainset.data.float().mean()
std = trainset.data.float().std()

tform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean / 255.0], std=[std / 255.0])
])
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=tform)
testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=tform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
fig, ax = plt.subplots(1, 5)

for i in np.arange(5):
    ax[i].imshow(trainset.data[i])
    ax[i].set_title(trainset.targets[i])

In [ ]:
trainset.data.shape
# trainset.targets[0]
trainset.targets

In [ ]:
unique_labels = np.unique(trainset.targets)

In [ ]:
# loss function - negative log likelihood
loss_fn = F.nll_loss

In [ ]:
def zero_grad(params):
    """
    Clear gradients as they may accumulate on successive backward calls

    Args:
        params: an iterator over tensors
            i.e., updating the weights and biases

    Returns:
        Nothing
    """

    for par in params:
        if not(par.grad is None):
            par.grad.data.zero_()


def gradient_update(loss, params, lr=1e-3):
    """
    Perform a gradient descent update on a given loss over a collection of parameters

    Args:
        loss: Tensor
            A scaler tensor containing the loss through which the gradient will be computed
        params: List of iterables
            Collection of parameters with respect to which we compute gradients
        lr: Float
            Scalar specifying the learning rate or step-size for the update
    
    Returns:
        Nothing
    """

    # Clean up gradients as Pytorch automatically accumulates gradients from successive backward calls

    zero_grad(params)

    # compute gradients on given objects
    loss.backward()

    with torch.no_grad():
        for par in params:
            par.data -= lr * par.grad.data


def print_params(model):
  """
  Lists the name and current value of the model's
  named parameters

  Args:
    model: an nn.Module inherited model
      Represents the ML/DL model

  Returns:
    Nothing
  """
  for name, param in model.named_parameters():
    if param.requires_grad:
      print(name, param.data)

In [ ]:
X_train = trainset.data.view(trainset.data.shape[0], -1).float()
y = trainset.targets

In [ ]:
net = Net(actv='LeakyReLU(0.1)', input_feature_num=X_train.shape[1], hidden_unit_nums=[100, 10, 5], output_feature_num=len(unique_labels)).to(DEVICE)

In [ ]:
optimizer = torch.optim.SGD(net.parameters(), lr=1e-1)

train_acc_list, test_acc_list = [], []
n_epochs = 10
criterion = F.nll_loss

In [ ]:
def train(model, train_loader, optimizer, criterion):
    model.train()
    for batch_id, (data, target) in enumerate(train_loader):
        data, target = data.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad()
        data = data.view(data.shape[0], -1).float()
        y_predicted = net(data.to(DEVICE))
        m = nn.LogSoftmax(dim=1)
        y_predicted = m(y_predicted)

        loss = criterion(y_predicted, target)

        loss.backward()
        optimizer.step()
    return model

def test(model, test_loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            data = data.view(data.shape[0], -1).float()
            y_predicted = net(data.to(DEVICE))
            m = nn.LogSoftmax(dim=1)
            y_predicted = m(y_predicted)
            test_loss += criterion(y_predicted, target, reduction='sum').item()
            pred = y_predicted.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    
    test_loss /= len(test_loader.dataset)
    return 100. * correct / len(test_loader.dataset)

In [ ]:
for epoch in np.arange(n_epochs):
    net_trained = train(net, trainloader, optimizer, criterion)
        
    # evaluate current model performance on training set and test set
    train_acc = test(net_trained, trainloader, criterion)
    test_acc = test(net_trained, testloader, criterion)

    train_acc_list.append(train_acc)
    test_acc_list.append(test_acc)

In [ ]:
fig, ax = plt.subplots(1, 1)

plt.plot(train_acc_list, 'r')
plt.plot(test_acc_list, 'b')

In [ ]:
y_predicted[0]

In [ ]:
loss = loss_fn(y_predicted, y)
gradient_update(loss, list(net.parameters()), lr=1e-1)
print('The net parameters after the updates are ')
print_params(net)

In [ ]:
y_predicted_labels = torch.argmax(y_predicted, axis=1)

In [ ]:
y_predicted_labels

In [ ]:
for i in np.arange(20):
    print(y_predicted_labels[i])